# Level Zero optimizer comparison

Compares SGD + momentum, AdamW, and Muon after verifying that model, data, split sizes, and training-token budgets are identical. Curves use **Bollinger-style across-seed envelopes** (mean ± 2 sample SD); final bars use 95% Student-t confidence intervals.

In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd
elif (cwd.parent / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd.parent
elif (cwd / "level_0_baseline" / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd / "level_0_baseline"
else:
    raise FileNotFoundError("Run from the repository or level_0_baseline tree")
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from level0_baseline.analysis import (
    load_metrics, load_spectral_metrics, load_test_results,
    plot_final_test_ci, plot_overlay_band, run_status_table,
    test_summary_table, validate_protocol_identity,
)
from level0_baseline.config import SUPPORTED_OPTIMIZERS, canonical_seeds, load_config

CONFIG = load_config(EXPERIMENT_ROOT / "configs" / "level0.yaml")
SEEDS = canonical_seeds(CONFIG)
OPTIMIZERS = SUPPORTED_OPTIMIZERS
ROOT = Path(os.environ.get("NANOGPT_LEVEL0_ROOT", "/tmp/nanogpt-level0-baselines"))
RESULTS_ROOT = Path(os.environ.get("NANOGPT_LEVEL0_RESULTS_ROOT", ROOT / "results"))
BAND_SIGMA = float(CONFIG["analysis"]["bollinger_sigma"])

In [ ]:
display(run_status_table(RESULTS_ROOT, optimizers=OPTIMIZERS, seeds=SEEDS))
protocol = validate_protocol_identity(RESULTS_ROOT, optimizers=OPTIMIZERS, seeds=SEEDS)
display(protocol)

In [ ]:
metrics = load_metrics(RESULTS_ROOT, optimizers=OPTIMIZERS, seeds=SEEDS)
spectral = load_spectral_metrics(RESULTS_ROOT, optimizers=OPTIMIZERS, seeds=SEEDS)
test_results = load_test_results(RESULTS_ROOT, optimizers=OPTIMIZERS, seeds=SEEDS)
test_summary = test_summary_table(test_results)

## Learning-curve overlays

In [ ]:
for metric in [
    "train_loss", "train_perplexity", "train_accuracy",
    "val_loss", "val_perplexity", "val_accuracy",
    "val_generalization_gap", "grad_norm_pre_clip",
    "update_to_weight_ratio", "tokens_per_sec",
]:
    plot_overlay_band(metrics, metric=metric, sigma=BAND_SIGMA, optimizers=OPTIMIZERS)
    plt.show()

## WeightWatcher alpha, ERG gap, and fit-quality overlays

In [ ]:
for metric in ["alpha_median", "ERG_gap_median", "D_median", "stable_rank_median"]:
    plot_overlay_band(spectral, metric=metric, sigma=BAND_SIGMA, optimizers=OPTIMIZERS)
    if metric == "alpha_median":
        plt.axhline(2.0, linestyle="--", linewidth=1.0)
    if metric == "ERG_gap_median":
        plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.show()

## Final and validation-selected held-out test comparisons

In [ ]:
display(test_summary[[
    "optimizer_label", "checkpoint", "metric", "n", "mean", "sd",
    "ci95_half_width", "ci95_lower", "ci95_upper"
]])
for checkpoint in ["final", "validation_selected"]:
    for metric in ["test_loss", "test_perplexity", "test_accuracy"]:
        plot_final_test_ci(test_summary, metric=metric, checkpoint=checkpoint, optimizers=OPTIMIZERS)
        plt.show()

Interpret all three-seed intervals cautiously: they reveal gross instability and optimizer separation, but they are not high-power estimates. The raw run directories and WeightWatcher layer tables remain the audit trail.